# Типы карт × MCC — заливка в DRP

Витрина для графика: **MCC-код, операции, оборот, доли платёжных систем**.  
Период на дашборде: те же фильтры `report_month` + `period_mode` (`month` / `ytd` / `quarter`).

| | |
|---|---|
| Источник | чекпоинты `qc_card_type_vs_mcc` — **Impala не сканируем** |
| Таблица | `sbx_da.tmp_shestopalov_acq_card_type_month` |
| Зерно | `report_month × mcc × payment_system` |
| `payment_system` | МИР / Visa / Mastercard / UnionPay / **Карта РСХБ** / прочие |
| Карта РСХБ | on-us: филиалы `*РФ`, Головной офис, ЦРМБ (Московский филиал) |

`final_df` и `tmp_shestopalov_acq_mcc_month` не трогаем.

Dataset: `sources/sql/vd_card_type_mcc_dashboard_period.sql`  
Инструкция: `HOW_TO_card_type_mcc_dashboard.md`


In [ ]:
import getpass
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd()
CKPT_DIR = DATA_DIR / 'qc_card_type_vs_mcc' / 'checkpoints'
OUT_DIR = DATA_DIR / 'qc_card_type_vs_mcc'
OUT_DIR.mkdir(parents=True, exist_ok=True)

period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'
period_months = pd.date_range(
    period_start,
    pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1),
    freq='MS',
)

run_drp_upload = True
drp_schema = 'sbx_da'
drp_table = 'tmp_shestopalov_acq_card_type_month'
drp_superset_grant_role = 'raisa_superset'
NOTEBOOK_REV = '2026-09-18-card-type-upload-v1'

print('rev', NOTEBOOK_REV)
print('CKPT_DIR', CKPT_DIR)
print('months', [m.strftime('%Y-%m') for m in period_months])
print('DRP', f'{drp_schema}.{drp_table}', 'upload', run_drp_upload)


## 0) Классификация — как в qc_card_type_vs_mcc v4


In [ ]:
EMPTY_LABELS = {'none', 'nan', 'null', 'unknown', 'empty'}
KNOWN_SCHEMES = ('UnionPay', 'Mastercard', 'Visa', 'МИР', 'JCB', 'Amex', 'СБП')


def _norm_desc(text):
    return '' if pd.isna(text) else str(text).strip().lower()


def classify_scheme_raw(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return None
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'master\s*card|mastercard|\bmaestro\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'(^|[^a-zа-я])мир([^a-zа-я]|$)|nspk\s*mir|\bmir\b', s):
        return 'МИР'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    if re.search(r'\bsbp\b|сбп', s):
        return 'СБП'
    return None


def is_rshb_issuer(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return False
    if re.search(r'рсхб|rshb|россельхоз', s):
        return True
    if re.search(r'\bрф\b|российский\s*филиал', s) or s.endswith('рф'):
        return True
    if re.search(r'головн\w*\s+офис|\bцрмб\b', s):
        return True
    return False


def classify_payment_system(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return 'прочие'
    raw = classify_scheme_raw(text)
    if raw in ('UnionPay', 'Mastercard', 'Visa', 'МИР'):
        return raw
    if raw in ('JCB', 'Amex', 'СБП'):
        return 'прочие'
    if is_rshb_issuer(text):
        return 'Карта РСХБ'
    return 'прочие'


def _ckpt_path(label):
    return CKPT_DIR / f'card_type_mcc_{label}.parquet'


def _load_ckpt(label):
    pq = _ckpt_path(label)
    gz = pq.with_suffix('.csv.gz')
    if pq.exists():
        return pd.read_parquet(pq)
    if gz.exists():
        return pd.read_csv(gz, compression='gzip')
    raise FileNotFoundError(f'нет чекпоинта {label}: {pq.name} / {gz.name}')


## 1) Собрать слой из чекпоинтов

Impala не нужен. Если месяца нет — остановимся, не будем тянуть trx.


In [ ]:
parts = []
missing = []
for m in period_months:
    label = m.strftime('%Y-%m')
    try:
        part = _load_ckpt(label)
    except FileNotFoundError as exc:
        missing.append(str(exc))
        continue
    if part is None or len(part) == 0:
        print(f'  {label}: empty')
        continue
    part = part.copy()
    if 'report_month' not in part.columns:
        part['report_month'] = label
    print(f'  {label}: rows={len(part):,}')
    parts.append(part)

if missing:
    raise RuntimeError('Сначала qc_card_type_vs_mcc.ipynb. Нет чекпоинтов: ' + '; '.join(missing))

raw = pd.concat(parts, ignore_index=True)
raw['report_month'] = raw['report_month'].astype(str).str[:7]
raw['card_type'] = raw['card_type'].astype(str).str.strip()
raw['mcc'] = raw['mcc'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)
raw['payment_system'] = raw['card_type'].map(classify_payment_system)

layer = (
    raw.groupby(['report_month', 'mcc', 'payment_system'], dropna=False)
    .agg(
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
    )
    .reset_index()
)

print('layer rows', len(layer), 'months', sorted(layer['report_month'].unique().tolist()))
print(layer.groupby('payment_system')[['trx_cnt', 'trx_sum']].sum().sort_values('trx_sum', ascending=False))
display(layer.head(12))

layer_csv = OUT_DIR / 'vd_acq_card_type_month_2026_01_2026_08.csv'
layer.to_csv(layer_csv, index=False, encoding='utf-8-sig')
print('CSV', layer_csv)


## 2) Заливка DRP


In [ ]:
if not run_drp_upload:
    print('SKIP DRP (run_drp_upload=False)')
else:
    if layer is None or len(layer) == 0:
        raise RuntimeError('layer пустой')

    target_fq = f'{drp_schema}.{drp_table}'
    print('Preparing DRP upload →', target_fq, 'rows=', len(layer))

    drp_user = input('DRP user: ').strip()
    drp_password = getpass.getpass('DRP password: ')
    drp_conn = connect(
        to='DRP',
        user_params={'user_name': drp_user, 'password': drp_password},
    )

    upload_df = layer.copy()
    upload_df.columns = [str(c).strip().lower() for c in upload_df.columns]
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)

    col_defs = [f'"{str(c).replace(chr(34), chr(34)+chr(34))}" TEXT' for c in upload_df.columns]
    create_sql = 'CREATE TABLE ' + target_fq + ' (' + ', '.join(col_defs) + ')'

    with drp_conn:
        drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
        drp_conn.execute(create_sql)
        drp_conn.write(table=target_fq, df=upload_df, mode='append')
        cnt_df = drp_conn.fetch(f'select count(*) as row_cnt from {target_fq}')
        try:
            drp_conn.execute(f'GRANT USAGE ON SCHEMA {drp_schema} TO {drp_superset_grant_role}')
            drp_conn.execute(f'GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role}')
            print(f'OK: GRANT SELECT ON {target_fq} TO {drp_superset_grant_role}')
        except Exception as grant_exc:
            print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
            print(f'  GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role};')

        chk = drp_conn.fetch(
            f'''
            SELECT
              NULLIF(SUBSTRING(BTRIM(CAST(report_month AS TEXT)) FROM 1 FOR 7), '') AS report_month,
              payment_system,
              COUNT(*) AS n
            FROM {target_fq}
            GROUP BY 1, 2
            ORDER BY 1, 2
            '''
        )

    row_cnt = int(pd.to_numeric(cnt_df.iloc[0, 0], errors='coerce')) if cnt_df is not None and len(cnt_df) else 0
    print('OK DRP rows =', row_cnt)
    display(chk)
    print('Superset: SQL Lab → vd_card_type_mcc_dashboard_period.sql → Save dataset')
    print('Чарт: Table, dimension mcc_label; колонки trx_cnt, trx_sum, share_*_sum_pct')
    print('Фильтры дашборда: report_month + period_mode. Chart-level filter по месяцу не ставить.')
